# Fermions (Normal ordering, Majoranas, Jordan-Wigner)

Walkthrough of `core/fermions.py`. Fermionic objects are defined in an encoding-agnostic way, and only become qubits when you explicitly ask via `.encode(...)`.

In [1]:
import warnings 
warnings.filterwarnings("ignore")

In [2]:
import numpy as np

from qaravan.applications import ground_state
from qaravan.core.fermions import (
    FermionicSum,
    FermionicEncoding,
    hopping,
    majorana_bilinear,
    c, 
    cdag, 
    number,
)
from qaravan.core.hamiltonians import FermionicHamiltonian, interleaved_mode_order
from qaravan.core import display_math
np.set_printoptions(precision=4, suppress=True)

## 1. Encoding (spinless) tight binding Hamiltonian

`FermionicSum` is a sum of `FermionicTerm`s (products of ladder operators). 

`FermionicHamiltonian(FermionicSum)` adds exactly one thing: an assertion, checked at construction, that the terms *are* Hermitian 

`FermionicEncoding` captures which encoding (`"jordan_wigner"`, for now) and which qubit each mode lands on (important later for spinful fermions). 

`encode()` returns  `PauliSum` for `FermionicSum` and `Hamiltonian` for `FermionicHamiltonian` 

In [3]:
num_modes = 4
t = 1.0

hopping_terms = []
for site in range(num_modes - 1):
    hopping_terms += hopping(site, site + 1, -t)

fsum = FermionicSum(hopping_terms, num_modes)
fsum

-1 c†_0 c_1 - 1 c†_1 c_0 - 1 c†_1 c_2 - 1 c†_2 c_1 - 1 c†_2 c_3 - 1 c†_3 c_2

In [4]:
fham= FermionicHamiltonian(hopping_terms, num_modes)
fham

-1 c†_0 c_1 - 1 c†_1 c_0 - 1 c†_1 c_2 - 1 c†_2 c_1 - 1 c†_2 c_3 - 1 c†_3 c_2

In [5]:
encoding = FermionicEncoding("jordan_wigner", num_modes=num_modes)
encoding

FermionicEncoding(name='jordan_wigner', num_modes=4, mode_order=None)

In [6]:
fsum_encoded = fsum.encode(encoding)
print(type(fsum_encoded).__name__)
fsum_encoded

PauliSum


-0.5 YYII - 0.5 XXII - 0.5 IYYI - 0.5 IXXI - 0.5 IIYY - 0.5 IIXX

In [7]:
fham_encoded = fham.encode(encoding)
print(type(fham_encoded).__name__)  
fham_encoded

Hamiltonian


Hamiltonian(num_sites=4)
-0.5 YYII - 0.5 XXII - 0.5 IYYI - 0.5 IXXI - 0.5 IIYY - 0.5 IIXX

## 2. `.to_matrix()` gives a matrix on the correct number of qubits

In [8]:
H = fham_encoded.to_matrix()
print("shape:", H.shape, "expected:", (2**encoding.num_qubits, 2**encoding.num_qubits))
assert H.shape == (2**encoding.num_qubits, 2**encoding.num_qubits)
assert np.allclose(H, H.conj().T)  # Hermitian, as guaranteed by jordan_wigner's pairing
print("Hermitian: OK")

shape: (16, 16) expected: (16, 16)
Hermitian: OK


## 3. Create, encode and measure a Majorana bilinear

In [9]:
majorana_obs = majorana_bilinear(0, 3)  # i * gamma_0 * gamma_1, on mode 0
majorana_obs

1j γ_0 γ̄_1

In [10]:
encoded_majorana_obs = encoding.encode_majorana_terms([majorana_obs])
print(type(encoded_majorana_obs).__name__)
encoded_majorana_obs

PauliSum


YYII

In [11]:
gs = ground_state(fham_encoded, fixed_particle_number = 2)
value = gs.expectation(encoded_majorana_obs)
assert abs(value.imag) < 1e-10  # Hermitian observable -> real expectation value

## 4. Spinful fermions and encoding ordering

Spinful models in `qaravan` share one labelling convention: **modes $[0, M)$ are up, $[M, 2M)$ are down**.

That convention lives in the *model*. The mode $\to$ qubit permutation lives in the *encoding*.
`spin_qubits()` composes the two. 

Note this is not a mere relabelling of the qubit register. For instance, `_majorana_pauli_string` builds the JW parity string over **qubit** indices, so a permuted `mode_order` is a genuinely different encoding with different Pauli weights.

In [12]:
for label, mode_order in [("identity", None), ("interleaved", interleaved_mode_order(3))]:
    spinful_encoding = FermionicEncoding("jordan_wigner", num_modes=6, mode_order=mode_order)
    up, down = spinful_encoding.spin_qubits()
    print(f"{label:12s} up qubits {up}   down qubits {down}")

identity     up qubits [0, 1, 2]   down qubits [3, 4, 5]
interleaved  up qubits [0, 2, 4]   down qubits [1, 3, 5]


## 5. Normal ordering, and how Hermiticity is decided

`FermionicTerm` preserves whatever operator order you wrote, so **the same operator can be spelled
several ways**: $n_a n_b$ and $n_b n_a$ are the same but hold different `ops`. `normal_order()` puts
every term into one canonical spelling -- creations left, mode indices ascending within each
block -- using

$$c_i c^\dagger_j = \delta_{ij} - c^\dagger_j c_i, \qquad
c^\dagger_i c^\dagger_j = -c^\dagger_j c^\dagger_i, \qquad c^\dagger_i c^\dagger_i = 0.$$

The $\delta_{ij}$ makes it a *sum*: one term can canonicalize into several.

In [13]:
forward = FermionicSum([number(0) @ number(1)], 2)
backward = FermionicSum([number(1) @ number(0)], 2)

display_math("as written:   ", forward, "   vs   ", backward)
display_math("normal-ordered:", forward.normal_order(), "  vs  ", backward.normal_order())
display_math("same operator recognised:",
      [t.ops for t in forward.normal_order().terms] == [t.ops for t in backward.normal_order().terms])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

The contraction is visible on the simplest case, $c_0 c^\dagger_0 = 1 - c^\dagger_0 c_0$ — one
term in, two out, one of them the identity (`ops == []`):

In [14]:
display_math(FermionicSum([c(0) @ cdag(0)], 1).normal_order())

<IPython.core.display.Math object>

### Hermiticity

With a canonical spelling available, `is_hermitian` is just: does $H$ normal-order to the same
thing as $H^\dagger$? No encoding is involved — this is decided entirely in fermionic language.

In [15]:
checks = {
    "n_0": [number(0)],
    "hopping + h.c.": [cdag(0) @ c(1), cdag(1) @ c(0)],
    "bare c_0": [c(0)],
    "one half of a hop": [cdag(0) @ c(1)],
    "i * n_0": [1j * number(0)],
}
for label, terms in checks.items():
    print(f"{label}:  {FermionicSum(terms, 2).is_hermitian()}")

n_0:  True
hopping + h.c.:  True
bare c_0:  False
one half of a hop:  False
i * n_0:  False


### Checking Hermiticity of pairing terms

In [16]:
pairing_ok = FermionicSum([cdag(0) @ cdag(1), c(1) @ c(0)], 2)
pairing_bad = FermionicSum([cdag(0) @ cdag(1), c(0) @ c(1)], 2)

print("c†_0 c†_1 + c_1 c_0  ->", pairing_ok.is_hermitian(), " (expected True)")
print("c†_0 c†_1 + c_0 c_1  ->", pairing_bad.is_hermitian(), " (expected False)")

c†_0 c†_1 + c_1 c_0  -> True  (expected True)
c†_0 c†_1 + c_0 c_1  -> False  (expected False)
